In [1]:
import pandas as pd
import json

In [2]:
df = pd.read_csv(r"C:\Users\konra\Desktop\vscode\projekt_zespolowy\smart-shopping-list-ML\ml\data\kaggle_prepared.csv")

In [3]:
number_of_users = df['user_id'].nunique()
print(f"Liczba unikalnych użytkowników: {number_of_users}")

Liczba unikalnych użytkowników: 206209


In [ ]:
min_orders = 9
valid_users = [uid for uid, grp in df.groupby("user_id") if len(grp) >= min_orders]
n_samples = 5
step = max(1, len(valid_users) // n_samples)
chosen = valid_users[::step][:n_samples]

In [ ]:
def _parse_categories(raw) -> list[str]:
    if isinstance(raw, list):
        return [str(c).strip().lower() for c in raw if c]
    if not isinstance(raw, str) or not raw.strip():
        return []
    try:
        parsed = ast.literal_eval(raw)
        if isinstance(parsed, list):
            return [str(c).strip().lower() for c in parsed if c]
    except Exception:
        pass
    return [raw.strip().lower()]
df2 = pd.read_csv(r'C:\Users\konra\Desktop\vscode\projekt_zespolowy\smart-shopping-list-ML\prepare_data_for_model\OpenFoodData.csv', usecols=["Id", "Name", "Categories"])
df2["Id"] = df2["Id"].astype(str)
df2 = df2.dropna(subset=["Name"])
_product_cache = {}
for _, row in df2.iterrows():
    pid = row["Id"]
    _product_cache[pid] = {
        "name": str(row["Name"]).strip(),
        "categories": _parse_categories(row.get("Categories", "")),
    }

recalls = []
for i, uid in enumerate(chosen, 1):
        print(f"\n{'─'*55}")
        print(f"[EVAL {i}/{len(chosen)}] user_id={uid}")
        id2name: dict[str, str] = {pid: info["name"] for pid, info in _product_cache.items()}
        user_df = df[df["user_id"] == uid].sort_values("order_id").reset_index(drop=True)
        history_rows = user_df.iloc[:-1]
        target_rows = user_df.iloc[-1]
        target = []
        for x in target_rows['off_product_id']:
            target.append(id2name[x])
        baskets = []
        history_baskets = {"history_baskets": []}
        for _, row in history_rows.iterrows():
            products = []
            for x in row["off_product_id"]:
                name = [id2name[str(x)]]
                products.append({"name":name, "id":x})
            days = int(row["days_since_prior_order"]) if pd.notna(row.get("days_since_prior_order")) else 0
            history_baskets["history_baskets"].append({"date": days, "products": products})
            #baskets.append({"days": days, "products": names})
        history_baskets = json.dumps(history_baskets, ensure_ascii=False) 
        print(history_baskets)
        import rag_pipeline
        import importlib
        importlib.reload(rag_pipeline)
        from rag_pipeline import stage1, stage2, stage3
        res_stage1 = stage1(history_baskets)
        print(res_stage1)
        res_stage2 = stage2(res_stage1)
        print(res_stage2)
        res_stage3 = stage3(res_stage2, final_k=10)
        print(res_stage3) 
        recall=0
        for r in res_stage3['recommended_products']:
            if r in target:
                recall+=1
        recalls.append(recall/len(target))
        print(recall/len(target))
res = sum(recalls)/len(recalls)
print(f"Recall@10: {res}")

[autoreload of rag_pipeline failed: Traceback (most recent call last):
  File "C:\Users\konra\AppData\Roaming\Python\Python311\site-packages\IPython\extensions\autoreload.py", line 280, in check
    elif self.deduper_reloader.maybe_reload_module(m):
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\konra\AppData\Roaming\Python\Python311\site-packages\IPython\extensions\deduperreload\deduperreload.py", line 533, in maybe_reload_module
    new_source_code = f.read()
                      ^^^^^^^^
  File "c:\Users\konra\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1250.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x81 in position 2028: character maps to <undefined>
]



───────────────────────────────────────────────────────
[EVAL 1/5] user_id=1
{"history_baskets": [{"date": 28, "products": [{"name": ["Soda oczyszczona spożywcza bez antyzbrylaczy – 1000g"], "id": "5902802802576"}, {"name": ["Beef Jerky – Tarczyński"], "id": "5908230536007"}, {"name": ["Pistachio Delight – Carte d’Or"], "id": "8711327672499"}, {"name": ["Ser włoszczowski – Auchan"], "id": "2228326001442"}, {"name": ["Jabłka – La-Sad – 1,5 kg"], "id": "20355944"}, {"name": ["Suszone jabłko o smaku ananasa – Crispy natural – 18g"], "id": "5900238965421"}, {"name": ["Milk chocolate-covered banana – Biedronka – 180 g"], "id": "5907554477454"}]}, {"date": 21, "products": [{"name": ["Soda oczyszczona spożywcza bez antyzbrylaczy – 1000g"], "id": "5902802802576"}, {"name": ["Beef Jerky – Tarczyński"], "id": "5908230536007"}, {"name": ["Pistachio Delight – Carte d’Or"], "id": "8711327672499"}, {"name": ["Ser włoszczowski – Auchan"], "id": "2228326001442"}, {"name": ["Almond Butter"], "id": "59

KeyboardInterrupt: 

In [14]:
import ast
def _parse_categories(raw) -> list[str]:
    if isinstance(raw, list):
        return [str(c).strip().lower() for c in raw if c]
    if not isinstance(raw, str) or not raw.strip():
        return []
    try:
        parsed = ast.literal_eval(raw)
        if isinstance(parsed, list):
            return [str(c).strip().lower() for c in parsed if c]
    except Exception:
        pass
    return [raw.strip().lower()]

def _parse_product_ids(raw) -> list[str]:
    if isinstance(raw, list):
        return [str(x) for x in raw if pd.notna(x)]
    if pd.isna(raw):
        return []
    if isinstance(raw, str):
        text = raw.strip()
        if not text:
            return []
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return [str(x) for x in parsed if pd.notna(x)]
        except Exception:
            pass
        return [text]
    return [str(raw)]

df2 = pd.read_csv(r'C:\Users\konra\Desktop\vscode\projekt_zespolowy\smart-shopping-list-ML\prepare_data_for_model\OpenFoodData.csv', usecols=["Id", "Name", "Categories"])
df2["Id"] = df2["Id"].astype(str)
df2 = df2.dropna(subset=["Name"])
_product_cache = {}
for _, row in df2.iterrows():
    pid = row["Id"]
    _product_cache[pid] = {
        "name": str(row["Name"]).strip(),
        "categories": _parse_categories(row.get("Categories", "")),
    }

recalls = []
for i, uid in enumerate(chosen, 1):
        print(f"\n{'─'*55}")
        print(f"[EVAL {i}/{len(chosen)}] user_id={uid}")
        id2name: dict[str, str] = {pid: info["name"] for pid, info in _product_cache.items()}
        user_df = df[df["user_id"] == uid].sort_values("order_id").reset_index(drop=True)
        history_rows = user_df.iloc[:-1]
        target_rows = user_df.iloc[-1]
        target = []
        target_ids = _parse_product_ids(target_rows["off_product_id"])
        for x in target_ids:
            if x in id2name:
                target.append(id2name[x])
        baskets = []
        history_baskets = {"history_baskets": []}
        for _, row in history_rows.iterrows():
            products = []
            row_product_ids = _parse_product_ids(row["off_product_id"])
            for x in row_product_ids:
                name = [id2name.get(str(x), str(x))]
                products.append({"name":name, "id":str(x)})
            days = int(row["days_since_prior_order"]) if pd.notna(row.get("days_since_prior_order")) else 0
            history_baskets["history_baskets"].append({"date": days, "products": products})
            #baskets.append({"days": days, "products": names})
        history_baskets = json.dumps(history_baskets, ensure_ascii=False) 
        print(history_baskets)
        import rag_pipeline
        import importlib
        importlib.reload(rag_pipeline)
        from rag_pipeline import stage1, stage2, stage3
        res_stage1 = stage1(history_baskets)
        print(res_stage1)
        res_stage2 = stage2(res_stage1)
        print(res_stage2)
        res_stage3 = stage3(res_stage2,history_baskets, final_k=10)
        print(res_stage3) 
        recall=0
        for r in res_stage3['recommended_products']:
            if r in target:
                recall+=1
        recalls.append(recall/len(target))
        print(recall/len(target))
res = sum(recalls)/len(recalls)
print(f"Recall@10: {res}")


───────────────────────────────────────────────────────
[EVAL 1/5] user_id=1
{"history_baskets": [{"date": 28, "products": [{"name": ["Soda oczyszczona spożywcza bez antyzbrylaczy – 1000g"], "id": "5902802802576"}, {"name": ["Beef Jerky – Tarczyński"], "id": "5908230536007"}, {"name": ["Pistachio Delight – Carte d’Or"], "id": "8711327672499"}, {"name": ["Ser włoszczowski – Auchan"], "id": "2228326001442"}, {"name": ["Jabłka – La-Sad – 1,5 kg"], "id": "20355944"}, {"name": ["Suszone jabłko o smaku ananasa – Crispy natural – 18g"], "id": "5900238965421"}, {"name": ["Milk chocolate-covered banana – Biedronka – 180 g"], "id": "5907554477454"}]}, {"date": 21, "products": [{"name": ["Soda oczyszczona spożywcza bez antyzbrylaczy – 1000g"], "id": "5902802802576"}, {"name": ["Beef Jerky – Tarczyński"], "id": "5908230536007"}, {"name": ["Pistachio Delight – Carte d’Or"], "id": "8711327672499"}, {"name": ["Ser włoszczowski – Auchan"], "id": "2228326001442"}, {"name": ["Almond Butter"], "id": "59

KeyboardInterrupt: 